# ML Fundamentals #6: K-Nearest Neighbors (KNN)

## Spotify Track Popularity Classification

This project implements a K-Nearest Neighbors classifier using Spotify track audio features.

The original Spotify popularity score is transformed into popularity categories, allowing KNN classification to be used to examine whether characteristics such as danceability, energy, acousticness, valence, tempo, and other audio features can distinguish different levels of track popularity.

The notebook covers:

- Data Loading
- Dataset Understanding
- Exploratory Data Analysis
- Popularity Class Construction
- Data Preprocessing
- Feature Scaling
- KNN Model Training
- Model Evaluation
- K-Value Analysis
- Neighbor and Distance Analysis
- Hyperparameter Tuning
- Model Serialization# ML Fundamentals #6: K-Nearest Neighbors (KNN)

## Spotify Track Popularity Classification

This project implements a K-Nearest Neighbors classifier using Spotify track audio features.

The original Spotify popularity score is transformed into popularity categories, allowing KNN classification to be used to examine whether characteristics such as danceability, energy, acousticness, valence, tempo, and other audio features can distinguish different levels of track popularity.

The notebook covers:

- Data Loading
- Dataset Understanding
- Exploratory Data Analysis
- Popularity Class Construction
- Data Preprocessing
- Feature Scaling
- KNN Model Training
- Model Evaluation
- K-Value Analysis
- Neighbor and Distance Analysis
- Hyperparameter Tuning
- Model Serialization

In [1]:
# Import libraries for data analysis, visualization, preprocessing, KNN modeling, and evaluation

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

In [2]:
# Create directories for datasets, models, figures, and tables

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

print("Project directories ready.")

Project directories ready.


## 2. Loading the Dataset

This project uses the Spotify Tracks Dataset, a large music dataset containing track metadata, Spotify popularity scores, genre information, and numerical audio characteristics.

The dataset contains approximately 114,000 tracks spanning 114 genres.

The original `popularity` variable ranges from 0 to 100. Popularity categories will be constructed later to formulate the problem as a K-Nearest Neighbors classification task.

In [3]:
# Load the Spotify Tracks dataset from an online CSV source

dataset_url = "https://github.com/yash-khandelwal-dev/spotify-audio-analysis/releases/download/Dataset/spotify.csv"

df = pd.read_csv(dataset_url)

df.head()

,track_genre_acoustic,track_genre_afrobeat,track_genre_alt-rock,track_genre_alternative,track_genre_ambient,track_genre_anime,track_genre_black-metal,track_genre_bluegrass,track_genre_blues,track_genre_brazil,...,key_cos,duration_log,duration_log_z,time_signature_class_boolean,loudness_yeo,is_instrumental,is_dance_hit,temp_zscore,track_genre,track_genre_le
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.866025,1.577830,0.186048,1.0,0.132593,0.0,0.0,-1.141849,acoustic,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.866025,1.250904,-0.942392,1.0,-1.658038,0.0,0.0,-1.489701,acoustic,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.000000,1.507132,-0.057979,1.0,-0.538749,0.0,0.0,-1.528296,acoustic,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.000000,1.473744,-0.173222,1.0,-1.803951,0.0,0.0,1.987849,acoustic,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.500000,1.461916,-0.214050,1.0,-0.528464,0.0,0.0,-0.073343,acoustic,0.0


In [4]:
# Display the dataset dimensions and available columns

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumns:")
print(df.columns.tolist())

Rows: 113999
Columns: 159

Columns:
['track_genre_acoustic', 'track_genre_afrobeat', 'track_genre_alt-rock', 'track_genre_alternative', 'track_genre_ambient', 'track_genre_anime', 'track_genre_black-metal', 'track_genre_bluegrass', 'track_genre_blues', 'track_genre_brazil', 'track_genre_breakbeat', 'track_genre_british', 'track_genre_cantopop', 'track_genre_chicago-house', 'track_genre_children', 'track_genre_chill', 'track_genre_classical', 'track_genre_club', 'track_genre_comedy', 'track_genre_country', 'track_genre_dance', 'track_genre_dancehall', 'track_genre_death-metal', 'track_genre_deep-house', 'track_genre_detroit-techno', 'track_genre_disco', 'track_genre_disney', 'track_genre_drum-and-bass', 'track_genre_dub', 'track_genre_dubstep', 'track_genre_edm', 'track_genre_electro', 'track_genre_electronic', 'track_genre_emo', 'track_genre_folk', 'track_genre_forro', 'track_genre_french', 'track_genre_funk', 'track_genre_garage', 'track_genre_german', 'track_genre_gospel', 'track_gen

In [5]:
# Save the raw dataset locally and export a preview table

df.to_csv(
    "data/raw/spotify_tracks.csv",
    index=False
)

df.head(10).to_csv(
    "outputs/tables/dataset_preview.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


### Dataset Loading Insights

- The dataset contains approximately 114,000 Spotify tracks.
- Each observation represents an individual music track.
- The dataset combines track metadata with numerical audio characteristics.
- Spotify popularity is represented by a score ranging from 0 to 100.
- The continuous popularity score will later be transformed into popularity categories for KNN classification.
- A local copy of the raw dataset has been stored to make subsequent analysis reproducible.

## 3. Understanding the Dataset

Before constructing popularity classes and training the KNN classifier, the dataset is examined to understand its structure, feature types, descriptive statistics, missing values, duplicate records, and popularity distribution.

In [6]:
# Examine dataset structure, feature types, and non-null observations

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 113999 entries, 0 to 113998
Columns: 159 entries, track_genre_acoustic to track_genre_le
dtypes: float64(158), str(1)
memory usage: 138.3 MB


In [7]:
# Generate descriptive statistics for the numerical Spotify features

dataset_summary = df.describe().T

dataset_summary

,count,mean,std,min,25%,50%,75%,max
track_genre_acoustic,113999.0,8.772007e-03,0.093248,0.000000,0.00000,0.000000,0.000000,1.000000
track_genre_afrobeat,113999.0,8.772007e-03,0.093248,0.000000,0.00000,0.000000,0.000000,1.000000
track_genre_alt-rock,113999.0,8.772007e-03,0.093248,0.000000,0.00000,0.000000,0.000000,1.000000
track_genre_alternative,113999.0,8.772007e-03,0.093248,0.000000,0.00000,0.000000,0.000000,1.000000
track_genre_ambient,113999.0,8.772007e-03,0.093248,0.000000,0.00000,0.000000,0.000000,1.000000
...,...,...,...,...,...,...,...,...
loudness_yeo,113999.0,-2.393428e-17,1.000004,-3.914959,-0.59212,0.065769,0.647973,28.711253
is_instrumental,113999.0,1.125975e-01,0.316102,0.000000,0.00000,0.000000,0.000000,1.000000
is_dance_hit,113999.0,2.017562e-03,0.044872,0.000000,0.00000,0.000000,0.000000,1.000000
temp_zscore,113999.0,4.567459e-16,1.000000,-4.074538,-0.76486,-0.004360,0.597876,4.043737


In [8]:
# Save descriptive statistics for later reference

dataset_summary.to_csv(
    "outputs/tables/dataset_summary.csv"
)

In [9]:
# Examine missing values across all dataset columns

missing_values = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values

,Missing Values,Missing Percentage
track_genre_acoustic,0,0.0
track_genre_afrobeat,0,0.0
track_genre_alt-rock,0,0.0
track_genre_alternative,0,0.0
track_genre_ambient,0,0.0
...,...,...
is_instrumental,0,0.0
is_dance_hit,0,0.0
temp_zscore,0,0.0
track_genre,1,0.0


In [10]:
# Save the missing-value analysis

missing_values.to_csv(
    "outputs/tables/missing_values.csv"
)

In [11]:
# Check the dataset for duplicate track records

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 1133


In [12]:
# Examine the distribution and range of Spotify popularity scores

print("Minimum popularity:", df["popularity"].min())
print("Maximum popularity:", df["popularity"].max())
print("Mean popularity:", round(df["popularity"].mean(), 2))
print("Median popularity:", df["popularity"].median())

Minimum popularity: -1.4902054888413832
Maximum popularity: 2.993122109473847
Mean popularity: -0.0
Median popularity: 0.0789591705689475


In [13]:
# Save the frequency distribution of the original popularity scores

popularity_distribution = (
    df["popularity"]
    .value_counts()
    .sort_index()
    .rename_axis("Popularity")
    .reset_index(name="Track Count")
)

popularity_distribution.to_csv(
    "outputs/tables/popularity_distribution.csv",
    index=False
)

popularity_distribution.head()

,Popularity,Track Count
0,-1.490205,16019
1,-1.445372,2140
2,-1.400539,1036
3,-1.355706,585
4,-1.310872,389


### Dataset Understanding Insights

- The dataset contains track metadata together with numerical Spotify audio features.
- The `popularity` attribute is a numerical score rather than a predefined classification label.
- Audio characteristics such as danceability, energy, loudness, acousticness, instrumentalness, valence, and tempo provide potential predictors for popularity classification.
- Missing values and duplicate records are examined before preprocessing to ensure data quality.
- The original popularity score is retained at this stage without modification.
- Popularity categories will be explicitly constructed in the preprocessing stage to formulate the KNN classification problem.